# B-03b: Temporal Fusion Transformer (follows B-03 config)

Fills the other gap in the original model roster (D-05: ... -> LSTM/**TFT** -> SARIMAX) -- a full TFT was
explicitly de-scoped at FC-02 time (`forecasting_scope.md`: "Full TFT/N-HiTS de-scoped -> VSN supplies native
importance", D-38) in favour of the lighter `LSTM_VSN` proxy. This notebook builds the canonical architecture
(Lim et al. 2021) for real: **Variable Selection Networks** (per-variable Gated Residual Network transform +
softmax-gated combination, separately for static/encoder/decoder inputs), **static covariate encoders** (4
context vectors from the tower one-hot: selection, enrichment, LSTM initial hidden/cell state), an **LSTM
encoder-decoder** with static-initialised state, a **gated locality-enhancement** skip, **static enrichment**,
**interpretable multi-head self-attention** (shared value projection across heads, causal-masked), and a final
gated position-wise feed-forward head -- all hand-rolled pure PyTorch (`src/models/forecasting_dl.py`, `TFT`
class), matching D-38's no-library convention.

**Same data/CV/harness as B-03/B-04**: `forecast_features_v2.csv` (hourly) / `forecast_daily_v2.csv` (daily),
Towers 4/9 main split test 2022-2023, Tower 2 expanding-window folds, partial-pooled training, leak-free
encoder/decoder split (decoder sees only known-future `fx_` drivers). Reuses `run_track`/`build_windows`/
`_eval_rows` unchanged -- the TFT's `forward(enc, dec, static) -> (B,H)` signature matches DLinear/LSTM/
LSTM_VSN exactly. `d_model=32, n_heads=4` kept modest (bounded-iteration norm, D-41) relative to a
research-scale TFT -- every architectural component is present, just sized for a single untuned run.

In [1]:
import sys, time, datetime; sys.path.insert(0, "../../src/models")
from pathlib import Path
import numpy as np, pandas as pd, torch
import forecasting_dl as fdl
HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
dev = fdl.get_device(); print("device:", dev, "| torch", torch.__version__)
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv"); print("matrix v2", m.shape, "| FX", len(fdl.FX))
W = {"A": fdl.build_windows(m, "A"), "B": fdl.build_windows(m, "B")}
print("enc/dec channels: A enc", W["A"][4]["enc"].shape[-1], "dec", W["A"][4]["dec"].shape[-1])
print("TFT param count (Track A sizing):", sum(p.numel() for p in fdl.TFT(
    fdl.TRACKS["A"]["L"], fdl.TRACKS["A"]["H"], W["A"][4]["enc"].shape[-1], W["A"][4]["dec"].shape[-1], 3).parameters()))

device: cuda | torch 2.11.0+cu128


matrix v2 (210459, 55) | FX 26


enc/dec channels: A enc 30 dec 26
TFT param count (Track A sizing): 424749


## 1  Train + evaluate TFT, both tracks (same 30 epochs as B-03/B-04's DL roster)

In [2]:
MODELS = ["TFT"]; EPOCHS = 30
ALL = []
for mdl in MODELS:
    for trk in ["A", "B"]:
        t0 = time.time()
        rows = fdl.run_track(trk, mdl, W, dev, epochs=EPOCHS)
        ALL += rows
        print(f"{mdl:9s} track {trk}: {len(rows):2d} rows  ({time.time()-t0:.0f}s)", flush=True)
R = pd.DataFrame(ALL); print("total rows", len(R))

TFT       track A: 15 rows  (1096s)


TFT       track B: 12 rows  (83s)


total rows 27


## 2  Results + save b03b_summary.csv

In [3]:
print("=== R2 (towers 4/9) ===")
print(R[R.tower.isin([4,9])].pivot_table(index=["track","tower"],columns="horizon",values="R2").round(3).to_string())
print("\n=== MASE (towers 4/9, <1 = beats persistence) ===")
print(R[R.tower.isin([4,9])].pivot_table(index=["track","tower"],columns="horizon",values="MASE").round(3).to_string())
R.to_csv(RESULTS/"b03b_summary.csv",index=False); print("\nsaved results/b03b_summary.csv")

=== R2 (towers 4/9) ===
horizon         1      3      6      7      12     14     24     48
track tower                                                        
A     4     -0.127    NaN -0.085    NaN -0.060    NaN -0.056 -0.062
      9     -0.173    NaN -0.022    NaN -0.034    NaN -0.040 -0.041
B     4     -1.078 -1.018    NaN -0.917    NaN -0.836    NaN    NaN
      9     -0.856 -0.765    NaN -0.644    NaN -0.623    NaN    NaN

=== MASE (towers 4/9, <1 = beats persistence) ===
horizon         1      3      6      7      12     14     24     48
track tower                                                        
A     4      1.248    NaN  0.916    NaN  0.957    NaN  0.924  0.872
      9      1.233    NaN  0.865    NaN  0.793    NaN  0.874  0.842
B     4      2.006  1.590    NaN  1.313    NaN  1.057    NaN    NaN
      9      1.581  1.245    NaN  1.022    NaN  1.001    NaN    NaN

saved results/b03b_summary.csv


## 3  Compare TFT vs B-03 (trees) and B-04 (DLinear/LSTM/LSTM-VSN)

In [4]:
b03=pd.read_csv(RESULTS/"b03_summary.csv"); b04=pd.read_csv(RESULTS/"b04_summary.csv")
key=["track","tower","horizon"]
best_b03=b03[b03.model.isin(["RF","XGB"])].sort_values("R2",ascending=False).drop_duplicates(key)[key+["model","R2"]].rename(columns={"model":"best_b03_model","R2":"R2_b03"})
best_b04=b04.sort_values("R2",ascending=False).drop_duplicates(key)[key+["model","R2"]].rename(columns={"model":"best_b04_model","R2":"R2_b04"})
j=R[key+["R2","MASE"]].rename(columns={"R2":"R2_tft","MASE":"MASE_tft"}).merge(best_b03,on=key).merge(best_b04,on=key)
print("=== TFT vs best-of-B03 (trees) vs best-of-B04 (DL), towers 4/9 ===")
print(j[j.tower.isin([4,9])].sort_values(["track","tower","horizon"]).to_string(index=False))
print("\n=== best daily R2 per tower: B03 (trees) vs B04 (DL) vs B03b (TFT) ===")
for t in [4,9]:
    a=b03[(b03.track=="B")&(b03.tower==t)&(b03.model.isin(["RF","XGB"]))]["R2"].max()
    b=b04[(b04.track=="B")&(b04.tower==t)]["R2"].max()
    c=R[(R.track=="B")&(R.tower==t)]["R2"].max()
    print(f"  Tower {t}:  B03(trees) {a:.3f}  |  B04(DL) {b:.3f}  |  B03b(TFT) {c:.3f}")

=== TFT vs best-of-B03 (trees) vs best-of-B04 (DL), towers 4/9 ===
track  tower  horizon  R2_tft  MASE_tft best_b03_model  R2_b03 best_b04_model  R2_b04
    A      4        1  -0.127    1.2484             RF   0.136        DLinear  -0.155
    A      4        6  -0.085    0.9158            XGB   0.048       LSTM_VSN  -0.097
    A      4       12  -0.060    0.9573             RF   0.078       LSTM_VSN  -0.173
    A      4       24  -0.056    0.9240             RF   0.049       LSTM_VSN  -0.121
    A      4       48  -0.062    0.8717            XGB   0.035       LSTM_VSN  -0.107
    A      9        1  -0.173    1.2331             RF   0.159        DLinear  -0.006
    A      9        6  -0.022    0.8646             RF   0.084       LSTM_VSN  -0.022
    A      9       12  -0.034    0.7931             RF   0.081        DLinear  -0.144
    A      9       24  -0.040    0.8738             RF   0.087           LSTM  -0.170
    A      9       48  -0.041    0.8423             RF   0.055           

## 4  Fix attempt -- regularised retrain (weight decay + early stopping)

The negative R2 was verified (not a bug -- see D-45/`b03a_b03b_results.md`): training loss converges cleanly
to a strong fit, but test-set generalisation is poor (weak positive correlation, dragged deeply negative by
occasional large overconfident spike-mispredictions) -- the signature of **overfitting**, not failure to learn.
Targeted fix (not a full HPO sweep, per the bounded-iteration norm, D-41): **AdamW weight decay** (1e-3) +
**early stopping on a held-out validation year**. Towers 4/9 main split: retrain on **2018-2020** (one year
shorter than the original 2018-2021), validate on **2021** (mirrors the precedent already set by FC-03/U-01's
conformal calibration, which also reserves 2021 as a held-out year), stop when validation loss stops improving
(patience=5 epochs), restore the best-validation-loss weights, then evaluate on the unchanged 2022-2023 test
period. Tower 2's expanding-window folds are left as the original (un-regularised) run -- the fold windows are
already short (~6 months train-to-test), too small to carve out a further validation slice without breaking
the existing CV design.

In [5]:
def run_main_regularized(track, model_name, W, device, epochs=30, weight_decay=1e-3, patience=5, seed=0):
    cfg = fdl.TRACKS[track]; unit = "h" if cfg["freq"] == "h" else "D"
    Wt = W[track]
    n_enc = Wt[4]["enc"].shape[-1]; n_dec = Wt[4]["dec"].shape[-1]
    cut2020 = pd.Timestamp("2020-12-31 23:59"); cut2021 = pd.Timestamp("2021-12-31 23:59")
    tr_parts = [fdl._subset(Wt[t], pd.DatetimeIndex(Wt[t]["ttime"][:, -1]) <= cut2020) for t in [2, 4, 9]]
    train = fdl._cat(tr_parts)
    val_parts = [fdl._subset(Wt[t], (pd.DatetimeIndex(Wt[t]["ttime"][:, -1]) > cut2020) &
                              (pd.DatetimeIndex(Wt[t]["ttime"][:, -1]) <= cut2021)) for t in [2, 4, 9]]
    val = fdl._cat(val_parts)
    tests = {t: fdl._subset(Wt[t], pd.DatetimeIndex(Wt[t]["otime"]).year.isin([2022, 2023])) for t in [4, 9]}
    mu, sdy = fdl._standardize(train, [val] + list(tests.values()))
    gclim, gl = fdl._clim({"ttime": train["ttime"], "y": train["y"]}, unit)
    model = fdl.build_model(model_name, cfg["L"], cfg["H"], n_enc, n_dec, 3)
    fdl.train_model(model, train, device, epochs=epochs, ch4_mu=mu, ch4_sd=sdy, seed=seed,
                     weight_decay=weight_decay, val_data=val, patience=patience)
    rows = []
    for t in [4, 9]:
        if len(tests[t]["enc"]) < 10: continue
        preds = fdl.predict(model, tests[t], device, mu, sdy)
        rows += fdl._eval_rows(track, model_name + "-Reg", t, tests[t], preds, gclim, gl, unit)
    return rows

ALL_REG = []
for trk in ["A", "B"]:
    t0 = time.time()
    rows = run_main_regularized(trk, "TFT", W, dev, epochs=30, weight_decay=1e-3, patience=5)
    ALL_REG += rows
    print(f"TFT-Reg   track {trk}: {len(rows):2d} rows  ({time.time()-t0:.0f}s)", flush=True)
R_REG = pd.DataFrame(ALL_REG); print("total TFT-Reg rows", len(R_REG))

TFT-Reg   track A: 10 rows  (133s)


TFT-Reg   track B:  8 rows  (7s)


total TFT-Reg rows 18


In [6]:
print("=== TFT (original, overfit) vs TFT-Reg (weight decay + early stopping) vs B-03 trees -- daily R2 ===")
key=["track","horizon","tower"]
orig=R[(R.tower.isin([4,9]))][key+["R2","MASE"]].rename(columns={"R2":"R2_orig","MASE":"MASE_orig"})
reg=R_REG[key+["R2","MASE"]].rename(columns={"R2":"R2_reg","MASE":"MASE_reg"})
b03best=b03[b03.model.isin(["RF","XGB"])&b03.tower.isin([4,9])].sort_values("R2",ascending=False).drop_duplicates(key)[key+["R2"]].rename(columns={"R2":"R2_b03"})
cmp=orig.merge(reg,on=key).merge(b03best,on=key).sort_values(["track","tower","horizon"])
print(cmp.to_string(index=False))
print("\n=== mean R2 delta (Reg - Orig), by track ===")
print(cmp.groupby("track")[["R2_orig","R2_reg"]].mean().round(3).to_string())

=== TFT (original, overfit) vs TFT-Reg (weight decay + early stopping) vs B-03 trees -- daily R2 ===
track  horizon  tower  R2_orig  MASE_orig  R2_reg  MASE_reg  R2_b03
    A        1      4   -0.127     1.2484   0.016    1.1614   0.136
    A        6      4   -0.085     0.9158   0.004    0.8617   0.048
    A       12      4   -0.060     0.9573   0.006    0.9024   0.078
    A       24      4   -0.056     0.9240   0.007    0.8705   0.049
    A       48      4   -0.062     0.8717   0.008    0.8110   0.035
    A        1      9   -0.173     1.2331   0.012    1.1902   0.159
    A        6      9   -0.022     0.8646   0.039    0.8679   0.084
    A       12      9   -0.034     0.7931   0.039    0.7890   0.081
    A       24      9   -0.040     0.8738   0.039    0.8616   0.087
    A       48      9   -0.041     0.8423   0.039    0.8165   0.055
    B        1      4   -1.078     2.0058   0.247    1.1822   0.362
    B        3      4   -1.018     1.5896   0.250    0.9445   0.345
    B        7 

## 4  Append to benchmarks.csv (B03b)

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B03b"]
RALL=pd.concat([R,R_REG],ignore_index=True)
rows=[]
for _,r in RALL.iterrows():
    is_reg=str(r["model"]).endswith("-Reg")
    rows.append({"replication":"B03b","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":("enriched_v2, canonical TFT + weight-decay/early-stopping fix (D-45)" if is_reg
                        else "enriched_v2, canonical TFT (VSN/GRN/static-encoders/interpretable-attention)"),
        "track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds","R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],
        "MBE":r["MBE"],"n_test":int(r["n_test"]),"skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "date":today,"notes":("B03b TFT regularised retrain (AdamW weight_decay=1e-3 + early stopping on 2021 validation year); fixes overfitting diagnosed in original TFT run (D-45)" if is_reg else
            "B03b canonical Temporal Fusion Transformer (fills D-05 TFT roster gap, de-scoped at FC-02/D-38); hand-rolled pure PyTorch; GPU; compare vs B03/B04")})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B03b rows ({len(R)} orig + {len(R_REG)} reg). Total {len(comb)}.")

Wrote 45 B03b rows (27 orig + 18 reg). Total 3737.
